# AyeAI Physical-AI Supply Network Optimization Lab

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.**  
**AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.**  
**License: GNU General Public License v3.0 only (GPL-3.0-only).**

This notebook is an optional browser-only computational workbench for the Physical-AI platform's supply-optimization companion module. The canonical platform remains static-first and does not require Colab, a backend, GPU access, or a proprietary service.

It demonstrates:
- constrained sourcing optimization using SciPy HiGHS (CPU);
- supplier concentration constraints and fail-closed qualification;
- network resilience using NetworkX vertex-disjoint paths and cut sets;
- deterministic event replay;
- Monte Carlo supply/service risk using NumPy, with optional PyTorch CUDA acceleration.

**Important:** a GPU is useful for large Monte Carlo/tensor workloads, not inherently for the linear-programming solve. The LP solver is CPU-based. Colab GPU availability is subject to runtime availability and quotas. A GPU runtime does not imply GPU utilization; this notebook reports the actual device used.

## Browser-only workflow
1. Open this notebook in Google Colab.
2. Upload `optimizer.py` from `additional-modules/supply-optimization/` when prompted.
3. Run the cells from top to bottom.
4. Optionally select **Runtime → Change runtime type → T4 GPU** for Monte Carlo acceleration.
5. Download the JSON result artifact from the final cell.

The notebook is intentionally self-contained in its sample problem and does not require access to the platform's private data. Use synthetic or approved, non-sensitive datasets only.

## Open in Colab after repository publication

After Grok commits this notebook, add the exact repository URL using the confirmed owner, repository and branch:

`https://colab.research.google.com/github/<OWNER>/<REPOSITORY>/blob/<BRANCH>/additional-modules/supply-optimization/notebooks/supply_chain_optimization.ipynb`

The Iteration 47 archive does not specify the GitHub owner/repository/branch, so this package does not invent a direct URL. Until the URL is filled, open https://colab.research.google.com/ and upload this notebook file.


## 1. Runtime and dependency setup

The required solver and graph packages are installed in the ephemeral Colab runtime. No installation is performed on the user's device or in the static platform. The optimization module uses SciPy's HiGHS LP solver; NetworkX is used for graph resilience. NumPy is the CPU simulation baseline. PyTorch CUDA is optional.

In [ ]:
%pip -q install --upgrade numpy scipy networkx

import os, sys, json, pathlib, platform, subprocess
print("Python:", platform.python_version())
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU not active. CPU execution remains fully supported.")
except Exception as exc:
    print("PyTorch/CUDA unavailable; Monte Carlo will use NumPy CPU:", type(exc).__name__)


## Optional alternative solver backends

The default SciPy HiGHS backend is installed in this notebook. The optimizer also exposes explicit optional adapters for OR-Tools GLOP and Pyomo + HiGHS. These are not silently substituted. Install only when you want to compare solvers:

```python
%pip install ortools pyomo highspy
```

Then rerun the backend discovery cell and set `problem["backend"]` to `ORTOOLS_GLOP` or `PYOMO_HIGHS`. Each solver must return a feasible optimal result for the same model before comparing allocations/objectives. GPU is not used by these LP solvers.

## 2. Load the optional optimization module

When this notebook is opened from GitHub, Colab does not automatically mount the repository checkout. Upload `optimizer.py` from the package ZIP using the browser upload dialog. If the file is already in the runtime working directory, no upload is needed.

In [ ]:
from pathlib import Path
if not Path("optimizer.py").exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if "optimizer.py" not in uploaded:
            raise RuntimeError("Upload optimizer.py from additional-modules/supply-optimization/")
    except ImportError:
        raise RuntimeError("Place optimizer.py beside this notebook or run in Google Colab and upload it.")
sys.path.insert(0, str(Path.cwd()))
from optimizer import (
    optimize_sourcing, safety_stock, simulate_uncertainty, available_backends,
    analyze_network_resilience, simulate_event_timeline
)
print("Optimization module loaded.")

print("Available solver backends:", json.dumps(available_backends(), indent=2))


## 3. Constrained sourcing optimization

The objective combines unit cost, lead time and carbon intensity with explicit weights. Only approved, lifecycle-eligible and export-cleared offers that meet due dates can be selected. The supplier share cap prevents the solver from silently concentrating all demand in one supplier. Infeasible constraints return HOLD.

In [ ]:
problem = {
    "demand": [
        {"part":"SERVO-24V","quantity":12,"due_days":7,"currency":"INR"},
        {"part":"MOTOR-DC","quantity":8,"due_days":10,"currency":"INR"}
    ],
    "offers": [
        {"id":"S1-SERVO","supplier_id":"S1","part":"SERVO-24V","capacity":8,"unit_cost":1200,"lead_days":3,"carbon_kg":2.1,"currency":"INR","approved":True,"export_status":"CLEAR","lifecycle_status":"ACTIVE"},
        {"id":"S2-SERVO","supplier_id":"S2","part":"SERVO-24V","capacity":8,"unit_cost":1280,"lead_days":5,"carbon_kg":1.7,"currency":"INR","approved":True,"export_status":"CLEAR","lifecycle_status":"ACTIVE"},
        {"id":"S3-SERVO","supplier_id":"S3","part":"SERVO-24V","capacity":10,"unit_cost":900,"lead_days":2,"carbon_kg":1.1,"currency":"INR","approved":False,"export_status":"CLEAR","lifecycle_status":"ACTIVE"},
        {"id":"S1-MOTOR","supplier_id":"S1","part":"MOTOR-DC","capacity":4,"unit_cost":850,"lead_days":4,"carbon_kg":1.8,"currency":"INR","approved":True,"export_status":"CLEAR","lifecycle_status":"ACTIVE"},
        {"id":"S2-MOTOR","supplier_id":"S2","part":"MOTOR-DC","capacity":8,"unit_cost":920,"lead_days":3,"carbon_kg":1.5,"currency":"INR","approved":True,"export_status":"CLEAR","lifecycle_status":"ACTIVE"}
    ],
    "weights":{"cost":1.0,"lead_time":20.0,"carbon":50.0},
    "max_supplier_share":0.75,
    "carbon_cap_kg":100
}
sourcing_result = optimize_sourcing(problem)
print(json.dumps(sourcing_result, indent=2))
assert sourcing_result["outcome"] == "OK"
assert sourcing_result["commitment"] == "OPTIMIZED_PLAN_NOT_PURCHASED"


## 4. Multi-tier network resilience

The model counts internally vertex-disjoint paths, not just the number of supplier names. Shared upstream materials, processes, sites, ownership or jurisdictions should be represented as shared graph nodes. Such common dependencies collapse apparently independent paths.

In [ ]:
nodes = [{"id":"SUP-A"},{"id":"SUP-B"},{"id":"RAW-ALLOY"},{"id":"MACHINE-SITE"},{"id":"PART"}]
edges = [
    {"from":"SUP-A","to":"RAW-ALLOY"},
    {"from":"SUP-B","to":"RAW-ALLOY"},
    {"from":"RAW-ALLOY","to":"MACHINE-SITE"},
    {"from":"MACHINE-SITE","to":"PART"}
]
network_result = analyze_network_resilience(nodes, edges, "SUP-A", "PART")
print(json.dumps(network_result, indent=2))
assert network_result["outcome"] == "OK"
assert network_result["vertex_disjoint_path_count"] == 1


## 5. Monte Carlo service-risk simulation (optional GPU)

The solver above remains CPU-based. This cell uses PyTorch CUDA when an available GPU is detected and requested; otherwise it uses the deterministic NumPy CPU path. Increase iterations only after checking runtime memory and quotas.

In [ ]:
mc_result = simulate_uncertainty(
    mean_daily_demand=10, demand_std=2,
    mean_lead_days=5, lead_std_days=1,
    supply_quantity=70, iterations=50000, seed=47, use_gpu=True
)
print(json.dumps(mc_result, indent=2))
assert mc_result["outcome"] == "OK"
print("Actual execution device:", mc_result["device"])


## 6. Deterministic logistics event replay

Events are replayed by logical time, not wall-clock time. This is useful for testing warehouse receipt, kitting, production, dispatch and reverse-logistics workflows against a known event trace.

In [ ]:
events = [
    {"id":"EV-04","time_days":4.0,"kind":"DELIVERY","entity_id":"ORDER-1","state":"DELIVERED"},
    {"id":"EV-01","time_days":0.0,"kind":"PROCUREMENT","entity_id":"PO-1","state":"CONFIRMED"},
    {"id":"EV-02","time_days":2.0,"kind":"RECEIPT","entity_id":"SHIP-1","state":"RECEIVED"},
    {"id":"EV-03","time_days":2.5,"kind":"KITTING","entity_id":"KIT-1","state":"READY"}
]
timeline = simulate_event_timeline(events)
print(json.dumps(timeline, indent=2))
assert timeline["outcome"] == "OK"
assert [e["id"] for e in timeline["timeline"]] == ["EV-01","EV-02","EV-03","EV-04"]


## 7. Inventory policy calculation

Safety stock uses an explicit normal approximation and a service-level input. It is not a substitute for multi-echelon stochastic inventory optimization. Review assumptions before operational use.

In [ ]:
policy = safety_stock(
    mean_daily_demand=10, demand_std=2,
    mean_lead_days=5, lead_std_days=1,
    service_level=0.975
)
print(json.dumps(policy, indent=2))
assert policy["outcome"] == "OK"


## 8. Export results

The JSON result file contains solver allocations, graph resilience, Monte Carlo risk and inventory policy. It is a research artifact, not an approved purchase order or shipment instruction.

In [ ]:
results = {
    "copyright":"Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.",
    "license":"GPL-3.0-only",
    "sourcing":sourcing_result,
    "network_resilience":network_result,
    "monte_carlo":mc_result,
    "event_timeline":timeline,
    "inventory_policy":policy
}
output_path = Path("logistics-optimization-results.json")
output_path.write_text(json.dumps(results, indent=2))
print("Wrote:", output_path.resolve())
try:
    from google.colab import files
    files.download(str(output_path))
except Exception:
    print("Download the JSON from the notebook file browser.")


## 9. Reproducibility, safety and limitations

- Solver: SciPy `linprog(method="highs")`, deterministic continuous LP for the supplied problem.
- Graph analysis: NetworkX directed graph and vertex-disjoint path/cut analysis.
- Monte Carlo: NumPy CPU baseline; PyTorch CUDA optional and explicitly reported.
- No live ERP, supplier, carrier, customs, sanctions, warehouse or robot system is contacted.
- No procurement, inventory reservation, dispatch, customs release, or safety authorization is performed.
- Export screening fields are workflow evidence inputs, not legal determinations.
- GPU availability is not guaranteed. The notebook runs without GPU.
- Persisted canonical data is not loaded by this notebook. Use synthetic, public, or explicitly approved data only.

**Grok integration:** keep this notebook in `additional-modules/supply-optimization/notebooks/`. Keep `optimizer.py` and `test_optimizer.py` beside the notebook. Add the repository-specific Colab URL only after the repository owner/name and target branch are confirmed. Do not embed credentials, tokens, private supplier data or secrets in the notebook.